In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        TRIAGEGEIST — SOTA Emergency Triage Prediction Notebook              ║
║        Target OOF kappa > 0.93  |  3-model ensemble + stacking             ║
╚══════════════════════════════════════════════════════════════════════════════╝

CELL 1 — SETUP & IMPORTS
"""

# ── Install / confirm deps ────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['lightgbm', 'xgboost', 'catboost', 'shap']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   capture_output=True)

import os, re, warnings, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (cohen_kappa_score, classification_report,
                              confusion_matrix, make_scorer)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import shap

SEED    = 42
N_FOLDS = 5
np.random.seed(SEED)

IS_KAGGLE = os.path.exists('/kaggle/input')
BASE = '/kaggle/input/competitions/triagegeist/'

print("Libraries loaded ✓")


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 2 — DATA LOADING"""

train_raw = pd.read_csv(BASE + 'train.csv')
test_raw  = pd.read_csv(BASE + 'test.csv')
cc        = pd.read_csv(BASE + 'chief_complaints.csv')
ph        = pd.read_csv(BASE + 'patient_history.csv')
sub_ref   = pd.read_csv(BASE + 'sample_submission.csv')

# Drop post-triage leakage columns (not available at triage time)
train_raw = train_raw.drop(columns=['ed_los_hours', 'disposition'], errors='ignore')

# Merge patient history (comorbidities)
train_raw = train_raw.merge(ph, on='patient_id', how='left')
test_raw  = test_raw.merge(ph, on='patient_id', how='left')

# Merge chief complaint raw text (richer than system category)
cc_slim   = cc[['patient_id', 'chief_complaint_raw']].drop_duplicates('patient_id')
train_raw = train_raw.merge(cc_slim, on='patient_id', how='left')
test_raw  = test_raw.merge(cc_slim, on='patient_id', how='left')

TARGET = 'triage_acuity'
y_all  = train_raw[TARGET].values

print(f"Train: {train_raw.shape}  Test: {test_raw.shape}")
print(f"Target distribution:\n{pd.Series(y_all).value_counts().sort_index()}")


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 3 — NLP: TF-IDF + SVD (BioBERT optional upgrade)"""

N_TEXT_COMPONENTS = 64   # Increase to 128 if GPU available

# Combine train + test texts for vocabulary (no leakage — unsupervised)
all_texts = pd.concat([
    train_raw['chief_complaint_raw'],
    test_raw['chief_complaint_raw']
], ignore_index=True).fillna('')

USE_BIOBERT = True  # Set True if dmis-lab/biobert-v1.1 is attached as Kaggle Model

if USE_BIOBERT:
    import torch
    from transformers import AutoTokenizer, AutoModel
    from tqdm import tqdm

    def get_biobert_embeddings(texts, batch_size=64):
        tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/vanle73/3-bio-bert/dmis-lab/biobert-large-cased-v1.1-squad_model")
        model     = AutoModel.from_pretrained("/kaggle/input/datasets/vanle73/3-bio-bert/dmis-lab/biobert-large-cased-v1.1-squad_model")
        device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model     = model.to(device).eval()
        embs = []
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = list(texts[i:i+batch_size])
            enc = tokenizer(batch, padding=True, truncation=True,
                            max_length=96, return_tensors='pt').to(device)
            with torch.no_grad():
                out = model(**enc)
            embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
        return np.vstack(embs)

    print("Extracting BioBERT embeddings …")
    raw_embs = get_biobert_embeddings(all_texts)
    svd = TruncatedSVD(n_components=N_TEXT_COMPONENTS, random_state=SEED)
    text_features = svd.fit_transform(raw_embs)
    print(f"Explained variance: {svd.explained_variance_ratio_.sum():.3f}")

else:
    # Fallback: TF-IDF with character + word n-grams → better for medical shorthand
    print("Using TF-IDF + SVD (fast baseline) …")
    tfidf_word = TfidfVectorizer(
        max_features=50_000, ngram_range=(1, 3),
        sublinear_tf=True, analyzer='word',
        token_pattern=r'(?u)\b\w+\b'
    )
    tfidf_char = TfidfVectorizer(
        max_features=30_000, ngram_range=(2, 4),
        sublinear_tf=True, analyzer='char_wb'
    )
    mat_word = tfidf_word.fit_transform(all_texts)
    mat_char = tfidf_char.fit_transform(all_texts)

    from scipy.sparse import hstack as sp_hstack
    mat_combined = sp_hstack([mat_word, mat_char])

    svd = TruncatedSVD(n_components=N_TEXT_COMPONENTS, random_state=SEED)
    text_features = svd.fit_transform(mat_combined)
    print(f"Text feature matrix: {text_features.shape}")
    print(f"Explained variance: {svd.explained_variance_ratio_.sum():.3f}")

text_cols = [f'txt_{i}' for i in range(N_TEXT_COMPONENTS)]
text_train = text_features[:len(train_raw)]
text_test  = text_features[len(train_raw):]
print(f"Text train: {text_train.shape}  Text test: {text_test.shape}")


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 4 — FEATURE ENGINEERING"""

HX_COLS = [c for c in ph.columns if c.startswith('hx_')]

CAT_COLS = [
    'arrival_mode', 'arrival_day', 'arrival_season', 'shift', 'age_group',
    'sex', 'language', 'insurance_type', 'transport_origin',
    'pain_location', 'mental_status_triage', 'chief_complaint_system',
]

VITALS_IMPUTE = [
    'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure', 'pulse_pressure',
    'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2',
]

def engineer(df, medians=None, le_dict=None, is_train=True):
    df = df.copy()

    # ── Pain score ─────────────────────────────────────────────────────────
    df['pain_not_recorded'] = (df['pain_score'] == -1).astype(np.int8)
    df['pain_clean']        = df['pain_score'].replace(-1, np.nan)

    # ── Missingness indicators (clinically informative) ────────────────────
    miss_cols = ['systolic_bp', 'diastolic_bp', 'respiratory_rate', 'spo2',
                 'temperature_c', 'heart_rate']
    for c in miss_cols:
        if c in df.columns:
            df[f'{c}_miss'] = df[c].isna().astype(np.int8)
    df['n_vitals_missing'] = df[[f'{c}_miss' for c in miss_cols
                                  if f'{c}_miss' in df.columns]].sum(axis=1).astype(np.int8)

    # ── Impute vitals with train medians ───────────────────────────────────
    if is_train:
        medians = {c: df[c].median() for c in VITALS_IMPUTE + ['pain_clean'] if c in df.columns}
    for c in VITALS_IMPUTE + ['pain_clean']:
        if c in df.columns:
            df[c] = df[c].fillna(medians.get(c, 0))

    # ── Binary ESI clinical thresholds ─────────────────────────────────────
    df['gcs_severe']       = (df['gcs_total'] < 9).astype(np.int8)
    df['gcs_moderate']     = ((df['gcs_total'] >= 9) & (df['gcs_total'] < 13)).astype(np.int8)
    df['spo2_critical']    = (df['spo2'] < 90).astype(np.int8)
    df['spo2_low']         = ((df['spo2'] >= 90) & (df['spo2'] < 94)).astype(np.int8)
    df['rr_high']          = (df['respiratory_rate'] > 25).astype(np.int8)
    df['rr_low']           = (df['respiratory_rate'] < 8).astype(np.int8)
    df['sbp_hypotensive']  = (df['systolic_bp'] < 90).astype(np.int8)
    df['sbp_hypertensive'] = (df['systolic_bp'] > 180).astype(np.int8)
    df['hr_tachy']         = (df['heart_rate'] > 100).astype(np.int8)
    df['hr_brady']         = (df['heart_rate'] < 50).astype(np.int8)
    df['temp_fever']       = (df['temperature_c'] > 38.3).astype(np.int8)
    df['temp_hypo']        = (df['temperature_c'] < 36.0).astype(np.int8)
    df['pain_severe']      = (df['pain_clean'] >= 8).astype(np.int8)
    df['news2_high']       = (df['news2_score'] >= 7).astype(np.int8)
    df['news2_medium']     = ((df['news2_score'] >= 5) & (df['news2_score'] < 7)).astype(np.int8)
    df['shock_high']       = (df['shock_index'] >= 1.0).astype(np.int8)
    df['map_critical']     = (df['mean_arterial_pressure'] < 65).astype(np.int8)

    # ── Composite severity index ────────────────────────────────────────────
    df['composite_sev'] = (
        df['news2_score'].fillna(0) * 1.5 +
        (15 - df['gcs_total'].fillna(15)) * 0.8 +
        df['pain_clean'].fillna(0) * 0.3 +
        df['shock_index'].fillna(0) * 2.0 +
        df['spo2_critical'] * 4 +
        df['sbp_hypotensive'] * 3 +
        df['gcs_severe'] * 5 +
        df['rr_high'] * 2
    ).astype(np.float32)

    # ── Vitals interaction features ─────────────────────────────────────────
    df['hr_to_sbp']   = (df['heart_rate'] / df['systolic_bp'].replace(0, np.nan)).fillna(0)
    df['rr_spo2_prod'] = df['respiratory_rate'] * (100 - df['spo2'])
    df['news2_gcs']    = df['news2_score'] * (15 - df['gcs_total'])

    # ── Mental status ordinal ───────────────────────────────────────────────
    ms_map = {'unresponsive': 0, 'drowsy': 1, 'agitated': 2, 'confused': 3, 'alert': 4}
    df['mental_enc']    = df['mental_status_triage'].str.lower().map(ms_map).fillna(4).astype(np.int8)
    df['mental_altered'] = (df['mental_enc'] < 4).astype(np.int8)

    # ── Comorbidity burden ─────────────────────────────────────────────────
    hx_present = [c for c in HX_COLS if c in df.columns]
    df['n_comorbidities_total'] = df[hx_present].fillna(0).sum(axis=1).astype(np.int8)
    df['high_comorbidity']      = (df['n_comorbidities_total'] >= 5).astype(np.int8)
    # High-risk syndrome combinations
    cardiac = [c for c in ['hx_heart_failure','hx_atrial_fibrillation','hx_coronary_artery_disease'] if c in df.columns]
    immune  = [c for c in ['hx_immunosuppressed','hx_hiv','hx_malignancy'] if c in df.columns]
    df['cardiac_burden']  = df[cardiac].fillna(0).sum(axis=1).astype(np.int8)
    df['immune_burden']   = df[immune].fillna(0).sum(axis=1).astype(np.int8)

    # ── NLP keyword features ────────────────────────────────────────────────
    txt = df['chief_complaint_raw'].fillna('').str.lower()
    PATS = {
        'kw_resus':   r'shock|arrest|unresponsive|\bcpr\b|resuscitat|apnea|unconscious',
        'kw_time':    r'stemi|sepsis|anaphylaxis|pulmonary.embolism|aortic|meningitis|eclampsia|torsion',
        'kw_high':    r'severe|worst|sudden|chest.pain|shortness.of.breath|\bsob\b|seizure|stroke|syncope|overdose|trauma',
        'kw_cardio':  r'chest|cardiac|palpitation|dyspnoea|dyspnea|angina',
        'kw_neuro':   r'headache|confusion|altered|vertigo|weakness|numbness',
        'kw_resp':    r'breath|respiratory|wheeze|cough|asthma|pneumonia',
        'kw_trauma':  r'trauma|fall|accident|lacerat|fracture|wound|burn',
        'kw_diaphor': r'diaphor|sweat',
        'kw_gi':      r'abdominal|nausea|vomit|diarrhoea|diarrhea|bleed',
        'kw_low':     r'mild|minor|routine|refill|follow.?up|chronic|dental|earache',
        'kw_worsening': r'worsening|deteriorat|progress|worsen',
        'kw_onset':   r'sudden.onset|acute.onset|new.onset|abrupt',
    }
    for name, pat in PATS.items():
        df[name] = txt.str.contains(pat, regex=True, na=False).astype(np.int8)
    df['kw_critical']  = ((df['kw_resus'] + df['kw_time']) > 0).astype(np.int8)
    df['kw_sev_score'] = (
        df['kw_resus'] * 4 + df['kw_time'] * 3 + df['kw_high'] * 2 +
        df['kw_cardio'] + df['kw_neuro'] + df['kw_onset'] * 2 +
        df['kw_worsening'] - df['kw_low'] * 2
    ).astype(np.float32)
    df['cc_len']         = df['chief_complaint_raw'].fillna('').str.len().astype(np.int16)
    df['cc_word_count']  = df['chief_complaint_raw'].fillna('').str.split().str.len().astype(np.int16)

    # ── Categorical label encoding ──────────────────────────────────────────
    if is_train:
        le_dict = {}
    for col in CAT_COLS:
        if col in df.columns:
            if is_train:
                le = LabelEncoder()
                df[col + '_enc'] = le.fit_transform(df[col].fillna('_UNK').astype(str))
                le_dict[col] = le
            else:
                le = le_dict.get(col)
                if le is not None:
                    known = set(le.classes_)
                    df[col + '_enc'] = df[col].fillna('_UNK').astype(str).apply(
                        lambda x: le.transform([x])[0] if x in known else -1
                    ).astype(np.int16)

    # ── Frequency encoding ─────────────────────────────────────────────────
    for col in ['site_id', 'triage_nurse_id']:
        if col in df.columns:
            freq = df[col].value_counts()
            df[col + '_freq'] = df[col].map(freq).fillna(0).astype(np.float32)

    # ── Cyclic time encoding ────────────────────────────────────────────────
    if 'arrival_hour' in df.columns:
        df['hour_sin'] = np.sin(2 * np.pi * df['arrival_hour'] / 24).astype(np.float32)
        df['hour_cos'] = np.cos(2 * np.pi * df['arrival_hour'] / 24).astype(np.float32)
    if 'arrival_month' in df.columns:
        df['month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12).astype(np.float32)
        df['month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12).astype(np.float32)
    if 'arrival_day' in df.columns:
        day_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
                   'Friday':4,'Saturday':5,'Sunday':6}
        df['day_num']   = df['arrival_day'].map(day_map).fillna(0)
        df['is_weekend'] = (df['day_num'] >= 5).astype(np.int8)
    if 'arrival_hour' in df.columns:
        df['is_night'] = ((df['arrival_hour'] >= 22) | (df['arrival_hour'] < 6)).astype(np.int8)

    return df, medians, le_dict


print("Engineering train …")
train_eng, MEDIANS, LE_DICT = engineer(train_raw, is_train=True)
print("Engineering test  …")
test_eng,  _,       _       = engineer(test_raw, medians=MEDIANS,
                                        le_dict=LE_DICT, is_train=False)

# ── Build feature list ─────────────────────────────────────────────────────
EXCLUDE_COLS = {
    'patient_id', TARGET, 'ed_los_hours', 'disposition',
    'chief_complaint_raw',
    # Original string columns (kept encoded versions)
    *CAT_COLS,
    'site_id', 'triage_nurse_id',
}
numeric_cols = train_eng.select_dtypes(include=[np.number]).columns.tolist()
FEATURES = [c for c in numeric_cols if c not in EXCLUDE_COLS]
print(f"\nFeature count: {len(FEATURES)}")
print(f"First 10: {FEATURES[:10]}")

X_train_base = train_eng[FEATURES].fillna(0).values.astype(np.float32)
X_test_base  = test_eng[FEATURES].fillna(0).values.astype(np.float32)

# Concatenate text features
X_train = np.hstack([X_train_base, text_train.astype(np.float32)])
X_test  = np.hstack([X_test_base,  text_test.astype(np.float32)])
print(f"Final X_train: {X_train.shape}  X_test: {X_test.shape}")


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 5 — MODEL PARAMETERS"""

try:
    import torch
    GPU = torch.cuda.is_available()
except Exception:
    GPU = False
print(f"GPU available: {GPU}")

LGBM_PARAMS = dict(
    objective         = 'multiclass',
    num_class         = 5,
    n_estimators      = 3000,
    learning_rate     = 0.035,
    max_depth         = 8,
    num_leaves        = 127,
    subsample         = 0.7,
    colsample_bytree  = 0.7,
    min_child_samples = 25,
    reg_alpha         = 0.05,
    reg_lambda        = 0.1,
    class_weight      = 'balanced',
    random_state      = SEED,
    verbose           = -1,
    n_jobs            = -1,
)

XGB_PARAMS = dict(
    objective              = 'multi:softprob',
    num_class              = 5,
    n_estimators           = 3000,
    learning_rate          = 0.035,
    max_depth              = 7,
    subsample              = 0.7,
    colsample_bytree       = 0.7,
    min_child_weight       = 25,
    reg_alpha              = 0.05,
    reg_lambda             = 0.1,
    eval_metric            = 'mlogloss',
    early_stopping_rounds  = 150,   # must be in constructor for xgboost>=2.0
    random_state           = SEED,
    n_jobs                 = -1,
    device                 = 'cuda' if GPU else 'cpu',
)

CAT_PARAMS = dict(
    iterations            = 4000,
    learning_rate         = 0.035,
    depth                 = 7,
    loss_function         = 'MultiClass',
    eval_metric           = 'MultiClass',
    random_seed           = SEED,
    early_stopping_rounds = 150,
    verbose               = 200,
    auto_class_weights    = 'Balanced',
    task_type             = 'GPU' if GPU else 'CPU',
)


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 6 — CROSS-VALIDATION"""

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y0  = y_all - 1   # 0-indexed classes for LGB / XGB

n_train = X_train.shape[0]
n_test  = X_test.shape[0]

oof_lgbm = np.zeros((n_train, 5), dtype=np.float32)
oof_xgb  = np.zeros((n_train, 5), dtype=np.float32)
oof_cat  = np.zeros((n_train, 5), dtype=np.float32)
tst_lgbm = np.zeros((n_test,  5), dtype=np.float32)
tst_xgb  = np.zeros((n_test,  5), dtype=np.float32)
tst_cat  = np.zeros((n_test,  5), dtype=np.float32)

fold_kappas = {'lgbm': [], 'xgb': [], 'cat': []}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_all)):
    print(f"\n{'━'*60}")
    print(f"  FOLD {fold+1}/{N_FOLDS}  |  train={len(tr_idx):,}  val={len(val_idx):,}")
    print(f"{'━'*60}")

    Xtr,  Xval  = X_train[tr_idx],  X_train[val_idx]
    ytr,  yval  = y0[tr_idx],       y0[val_idx]
    ytr1, yval1 = y_all[tr_idx],    y_all[val_idx]

    # ── LightGBM ────────────────────────────────────────────────────────────
    print("\n  LightGBM …")
    lgbm_mdl = lgb.LGBMClassifier(**LGBM_PARAMS)
    lgbm_mdl.fit(
        Xtr, ytr,
        eval_set=[(Xval, yval)],
        callbacks=[lgb.early_stopping(150, verbose=False),
                   lgb.log_evaluation(500)]
    )
    p_val = lgbm_mdl.predict_proba(Xval)
    p_tst = lgbm_mdl.predict_proba(X_test)
    oof_lgbm[val_idx] = p_val
    tst_lgbm += p_tst / N_FOLDS
    k = cohen_kappa_score(yval1, p_val.argmax(1)+1, weights='linear')
    fold_kappas['lgbm'].append(k)
    print(f"  → lgbm  kappa={k:.4f}  iter={lgbm_mdl.best_iteration_}")

    # ── XGBoost ─────────────────────────────────────────────────────────────
    print("\n  XGBoost …")
    xgb_mdl = xgb.XGBClassifier(**XGB_PARAMS)
    xgb_mdl.fit(
        Xtr, ytr,
        eval_set=[(Xval, yval)],
        verbose=500
    )
    p_val = xgb_mdl.predict_proba(Xval)
    p_tst = xgb_mdl.predict_proba(X_test)
    oof_xgb[val_idx] = p_val
    tst_xgb += p_tst / N_FOLDS
    k = cohen_kappa_score(yval1, p_val.argmax(1)+1, weights='linear')
    fold_kappas['xgb'].append(k)
    print(f"  → xgb   kappa={k:.4f}  iter={xgb_mdl.best_iteration}")

    # ── CatBoost ────────────────────────────────────────────────────────────
    print("\n  CatBoost …")
    cat_mdl = CatBoostClassifier(**CAT_PARAMS)
    cat_mdl.fit(
        Xtr, ytr,
        eval_set=(Xval, yval),
        use_best_model=True,
    )
    p_val = cat_mdl.predict_proba(Xval)
    p_tst = cat_mdl.predict_proba(X_test)
    oof_cat[val_idx] = p_val
    tst_cat += p_tst / N_FOLDS
    k = cohen_kappa_score(yval1, p_val.argmax(1)+1, weights='linear')
    fold_kappas['cat'].append(k)
    print(f"  → cat   kappa={k:.4f}  iter={cat_mdl.best_iteration_}")

    del lgbm_mdl, xgb_mdl, cat_mdl
    gc.collect()

# OOF summary
print("\n" + "═"*60)
for name, probs in [('lgbm', oof_lgbm), ('xgb', oof_xgb), ('cat', oof_cat)]:
    k    = cohen_kappa_score(y_all, probs.argmax(1)+1, weights='linear')
    kfds = [round(x, 4) for x in fold_kappas[name]]
    print(f"  {name.upper():5s}  OOF kappa={k:.4f}  folds={kfds}")
print("═"*60)


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 7 — ENSEMBLE: GRID-SEARCH BLEND + STACKING"""

# ── A: Grid-search blend weights ────────────────────────────────────────────
print("Grid-searching blend weights …")
best_k, best_w = -1, (0.4, 0.3, 0.3)
for a in np.arange(0.1, 0.7, 0.05):
    for b in np.arange(0.1, 0.6, 0.05):
        g = round(1.0 - a - b, 2)
        if not (0.05 <= g <= 0.6):
            continue
        blend = a * oof_lgbm + b * oof_xgb + g * oof_cat
        k = cohen_kappa_score(y_all, blend.argmax(1)+1, weights='linear')
        if k > best_k:
            best_k, best_w = k, (a, b, g)

wa, wb, wg = best_w
print(f"Best blend  lgbm={wa:.2f}  xgb={wb:.2f}  cat={wg:.2f}  kappa={best_k:.4f}")
oof_blend  = wa * oof_lgbm + wb * oof_xgb + wg * oof_cat
tst_blend  = wa * tst_lgbm + wb * tst_xgb + wg * tst_cat

# ── B: Stacking (logistic meta-learner) ─────────────────────────────────────
print("\nTraining stack meta-learner …")
meta_tr = np.hstack([oof_lgbm, oof_xgb, oof_cat])
meta_te = np.hstack([tst_lgbm, tst_xgb, tst_cat])

scaler     = StandardScaler()
meta_tr_s  = scaler.fit_transform(meta_tr)
meta_te_s  = scaler.transform(meta_te)

# Use nested CV to get unbiased OOF stack predictions
oof_stack = np.zeros((n_train, 5), dtype=np.float32)
tst_stack = np.zeros((n_test,  5), dtype=np.float32)

inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED+1)
for tr_i, val_i in inner_skf.split(meta_tr_s, y_all):
    lr = LogisticRegression(C=1.0, max_iter=1000, multi_class='multinomial',
                             solver='lbfgs', class_weight='balanced', random_state=SEED)
    lr.fit(meta_tr_s[tr_i], y_all[tr_i])
    oof_stack[val_i] = lr.predict_proba(meta_tr_s[val_i])
    tst_stack += lr.predict_proba(meta_te_s) / 5

stack_kappa = cohen_kappa_score(y_all, oof_stack.argmax(1)+1, weights='linear')
print(f"Stack OOF kappa: {stack_kappa:.4f}")

# ── Final selection ──────────────────────────────────────────────────────────
if stack_kappa >= best_k:
    final_oof  = oof_stack
    final_test = tst_stack
    method     = f'stack (kappa={stack_kappa:.4f})'
else:
    final_oof  = oof_blend
    final_test = tst_blend
    method     = f'blend (kappa={best_k:.4f})'
print(f"\nSelected: {method}")


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 8 — EVALUATION"""

final_preds = final_oof.argmax(1) + 1
final_kappa = cohen_kappa_score(y_all, final_preds, weights='linear')
print(f"\n{'═'*55}")
print(f"  FINAL OOF Linear Weighted Kappa: {final_kappa:.4f}")
print(f"{'═'*55}\n")
print(classification_report(y_all, final_preds,
                              target_names=[f'ESI-{i}' for i in range(1,6)]))

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cm      = confusion_matrix(y_all, final_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels  = [f'ESI-{i}' for i in range(1, 6)]

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix (Normalized)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Confusion Matrix (Raw Counts)')

plt.suptitle(f'Final Ensemble — OOF kappa = {final_kappa:.4f}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_final.png', dpi=120)
plt.show()

# Probability calibration check
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
ESI_COLORS = ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd']
for cls in range(5):
    mask = (y_all == cls+1)
    axes[cls].hist(final_oof[mask, cls], bins=30, color=ESI_COLORS[cls], alpha=0.7, label='Correct class')
    axes[cls].hist(final_oof[~mask, cls], bins=30, color='gray', alpha=0.4, label='Other')
    axes[cls].set_title(f'ESI-{cls+1} prob. dist')
    axes[cls].legend(fontsize=7)
plt.suptitle('Predicted Probability Distributions per ESI Class', fontsize=12)
plt.tight_layout()
plt.savefig('prob_distributions.png', dpi=120)
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 9 — SHAP ANALYSIS (LightGBM final model)"""

print("Fitting final LightGBM on full train for SHAP …")
final_lgbm = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'n_estimators': 1500})
final_lgbm.fit(X_train, y0)

sample_idx  = np.random.choice(len(X_train), 3000, replace=False)
X_shap      = X_train[sample_idx]
feat_names  = FEATURES + [f'txt_{i}' for i in range(N_TEXT_COMPONENTS)]

explainer   = shap.TreeExplainer(final_lgbm)
shap_values = explainer.shap_values(X_shap)  # list of 5 arrays

# Global importance (mean |SHAP| across classes)
if isinstance(shap_values, list):
    mean_abs = np.stack([np.abs(sv).mean(axis=0) for sv in shap_values]).mean(axis=0)
else:
    mean_abs = np.abs(shap_values).mean(axis=(0, 2))

importance_df = (pd.DataFrame({'feature': feat_names, 'importance': mean_abs})
                 .sort_values('importance', ascending=False)
                 .reset_index(drop=True))

print(f"\nTop 20 features (global mean |SHAP|):")
print(importance_df.head(20).to_string(index=False))

# Bar plot
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance_df.head(20).sort_values('importance')
ax.barh(top20['feature'], top20['importance'], color='steelblue', alpha=0.85)
ax.set_title('Top 20 Global SHAP Importances (LightGBM)', fontweight='bold')
ax.set_xlabel('Mean |SHAP|')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=120)
plt.show()

# Beeswarm for ESI-1 (most critical)
sv_esi1 = shap_values[0] if isinstance(shap_values, list) else shap_values[:, :, 0]
plt.figure(figsize=(11, 8))
shap.summary_plot(sv_esi1, X_shap, feature_names=feat_names,
                  max_display=20, show=False)
plt.title('SHAP Beeswarm — ESI-1 (Immediate) Class', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm_esi1.png', dpi=120)
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 10 — CLINICAL EQUITY AUDIT"""

audit = train_raw[['sex', 'age_group', 'language', 'insurance_type',
                    'arrival_mode']].copy()
audit['y_true']     = y_all
audit['y_pred']     = final_preds
audit['error']      = audit['y_pred'] - audit['y_true']
audit['undertriage'] = (audit['y_pred'] > audit['y_true']).astype(int)
audit['overtriage']  = (audit['y_pred'] < audit['y_true']).astype(int)
audit['exact']       = (audit['y_pred'] == audit['y_true']).astype(int)

print("\n━━━ EQUITY AUDIT ━━━\n")
for col in ['sex', 'age_group', 'language', 'insurance_type']:
    grp = audit.groupby(col).agg(
        n           = ('y_true',      'count'),
        undertriage = ('undertriage', 'mean'),
        overtriage  = ('overtriage',  'mean'),
        accuracy    = ('exact',       'mean'),
        mean_error  = ('error',       'mean'),
    ).round(4)
    print(f"── {col.upper()} ──")
    print(grp.to_string())
    print()

# Undertriage heatmap: sex × insurance_type
pivot = audit.pivot_table(
    values='undertriage', index='sex', columns='insurance_type', aggfunc='mean'
).fillna(0) * 100
fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Oranges', ax=ax,
            cbar_kws={'label': 'Undertriage Rate (%)'})
ax.set_title('Undertriage Rate (%) by Sex × Insurance Type', fontweight='bold')
plt.tight_layout()
plt.savefig('equity_heatmap.png', dpi=120)
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
"""CELL 11 — GENERATE SUBMISSION"""

test_preds = final_test.argmax(1) + 1

submission = pd.DataFrame({
    'patient_id':    test_raw['patient_id'],
    'triage_acuity': test_preds.astype(int)
})

assert list(submission.columns) == list(sub_ref.columns), "Column mismatch!"
assert submission['triage_acuity'].between(1, 5).all(), "Out-of-range predictions!"
assert len(submission) == len(sub_ref), f"Row mismatch: {len(submission)} vs {len(sub_ref)}"

print("Submission validation passed ✓")
print(f"\nPrediction distribution:")
print(submission['triage_acuity'].value_counts().sort_index())
print(f"\nTrain distribution (reference):")
print(pd.Series(y_all).value_counts().sort_index())

kappa_tag = f"{final_kappa:.4f}".replace('.', 'p')
fname = f'submission_triagegeist_sota_{kappa_tag}.csv'
submission.to_csv(fname, index=False)
print(f"\n✅ Saved: {fname}")
print(submission.head(10))